In [1]:
!pip install pyspark

In [16]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark import StorageLevel
import pandas as pd
import time

In [3]:
spark = SparkSession.builder \
    .appName("NYC Taxi Task4") \
    .getOrCreate()

In [4]:
df = spark.read.parquet("yellow_tripdata_2024-01.parquet")

Check Initial Partitions

In [5]:
print("Initial Partitions:")
print(df.rdd.getNumPartitions())

Initial Partitions:
2


Benchmark 1 - Measure Execution Time Without Cache

In [6]:
start = time.time()

df.groupBy("payment_type") \
  .agg(avg("fare_amount")) \
  .show()

without_cache_time = time.time() - start

print("Without Cache:", without_cache_time)

+------------+------------------+
|payment_type|  avg(fare_amount)|
+------------+------------------+
|           1|18.557432202727387|
|           3| 6.752568760524564|
|           2|17.866037304953625|
|           4|1.3348886934888904|
|           0|20.016193904200065|
+------------+------------------+

Without Cache: 4.063378572463989


Apply Cache

In [7]:
df.cache()

df.count()

2964624

Benchmark 2 - Measure Execution Time With Cache

In [8]:
start = time.time()

df.groupBy("payment_type") \
  .agg(avg("fare_amount")) \
  .show()

with_cache_time = time.time() - start

print("With Cache:", with_cache_time)

+------------+------------------+
|payment_type|  avg(fare_amount)|
+------------+------------------+
|           1|18.557432202727387|
|           3| 6.752568760524564|
|           2|17.866037304953625|
|           4|1.3348886934888904|
|           0|20.016193904200065|
+------------+------------------+

With Cache: 1.559370994567871


Compare Cache Performance

In [9]:
print("Without Cache:", without_cache_time)

print("With Cache:", with_cache_time)

improvement = (
    (without_cache_time - with_cache_time)
    /
    without_cache_time
) * 100

print("Improvement %:", improvement)

Without Cache: 4.063378572463989
With Cache: 1.559370994567871
Improvement %: 61.62378260457565


Persist Dataset

In [10]:
from pyspark import StorageLevel

df.persist(StorageLevel.MEMORY_AND_DISK)

df.count()

print("Dataset Persisted")

Dataset Persisted


Repartition Dataset

In [11]:
repartitioned_df = df.repartition(4)

print(
    "Partitions After Repartition:"
)

print(
    repartitioned_df.rdd.getNumPartitions()
)

Partitions After Repartition:
4


Benchmark 3 - Measure Performance After Repartition

In [12]:
start = time.time()

repartitioned_df.groupBy(
    "payment_type"
).agg(
    avg("tip_amount")
).show()

repartition_time = time.time() - start

print(
    "Execution Time:",
    repartition_time
)

+------------+--------------------+
|payment_type|     avg(tip_amount)|
+------------+--------------------+
|           0|   1.545956750046391|
|           1|   4.169670627492557|
|           3|0.014559881614532841|
|           2|0.002296016994883...|
|           4| 0.04212511795487691|
+------------+--------------------+

Execution Time: 4.612051486968994


Spark Configuration

In [13]:
spark.sparkContext.getConf().getAll()

[('spark.app.startTime', '1781363890823'),
 ('spark.rdd.compress', 'True'),
 ('spark.hadoop.fs.s3a.vectored.read.min.seek.size', '128K'),
 ('spark.app.name', 'NYC Taxi Task4'),
 ('spark.driver.host', '89c131de5be9'),
 ('spark.executor.extraJavaOptions',
  '-Djava.net.preferIPv6Addresses=false -XX:+IgnoreUnrecognizedVMOptions --add-modules=jdk.incubator.vector --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.invoke=ALL-UNNAMED --add-opens=java.base/java.lang.reflect=ALL-UNNAMED --add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/java.net=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED --add-opens=java.base/java.util.concurrent=ALL-UNNAMED --add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED --add-opens=java.base/jdk.internal.ref=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED --add-opens=java.base/sun.nio.cs=ALL-UNNAMED --add-opens=java.base/sun.security.action=ALL-UNNAMED --add-o

Application Details

In [14]:
print(
    "Application ID:",
    spark.sparkContext.applicationId
)

print(
    "Spark Version:",
    spark.version
)

Application ID: local-1781363893567
Spark Version: 4.0.2


Scalability Table

In [17]:
scalability_df = pd.DataFrame(
    {
        "Metric":[
            "Without Cache",
            "With Cache",
            "Repartitioned"
        ],
        "Execution_Time":[
            without_cache_time,
            with_cache_time,
            repartition_time
        ]
    }
)

scalability_df

,Metric,Execution_Time
0,Without Cache,4.063379
1,With Cache,1.559371
2,Repartitioned,4.612051


Export For Tableau

In [18]:
scalability_df.to_csv(
    "scalability_metrics.csv",
    index=False
)

print(
    "Scalability Metrics Saved"
)

Scalability Metrics Saved
